<h1>Important</h1>

- The following notebook has only personal learning purposes with no further intention. This was developed using AI tools combined with multiple iterations to refine the code given at first it does generate many errors, from documentation error and more.
- The key intention of this notebook is to show to use a model available from Hugging Face in combination with techniques from the same library to do fine-tuning of the model and show how it works.
- This is not a comercial or industry code to be used, it is just a personal academic learning of how to use different python libraries with ideas of Reinforcement Learning and LLMs.

Key considerations when running this:
- The machine that was used had installed cuda nvidia with 8 GB of capacity, and the idea was to constraint the dataset size and memory usage when training the model.
- Do not load more than 1 model if the CUDA capacity is small because it will lead to potential crushing.
- Try to use cuda and not cpu because is much faster when running.

_____

# Complete LLM Fine-tuning

Progressive Training with Simple Cooking Domain

This guide demonstrates five cutting-edge fine-tuning techniques:
- **Supervised Fine-Tuning (SFT)**: Foundation adaptation with cooking examples
- **Kahneman-Tversky Optimization (KTO)**: Behavioral economics preference learning
- **Odds Ratio Preference Optimization (ORPO)**: Single-step SFT+preference optimization
- **Group Relative Policy Optimization (GRPO)**: Interactive learning with human feedback

Considerations
- **Domain Focus**: Simple cooking advice for clear, demonstrable improvements
- **Strategy**: Single model progression with memory optimization

In [1]:
# =============================================================================
# SETUP AND ENHANCED CONFIGURATION
# =============================================================================

import warnings

warnings.filterwarnings("ignore")

import subprocess
import sys
import torch
import gc
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset

# Enhanced Model Configuration
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
TEMPERATURE = 0.8
MAX_LENGTH = 512  # Increased for richer responses
MAX_NEW_TOKENS = 200  # Longer responses

# Optimized Training Parameters for Deep Learning
BATCH_SIZE_SFT = 1
BATCH_SIZE_KTO = 2
BATCH_SIZE_ORPO = 1
BATCH_SIZE_GRPO = 1

# Enhanced gradient accumulation for better training
GRAD_ACCUM_SFT = 16  # Increased significantly
GRAD_ACCUM_KTO = 8
GRAD_ACCUM_ORPO = 12
GRAD_ACCUM_GRPO = 8

# Optimized learning rates for each technique
LEARNING_RATE_SFT = 4e-5
LEARNING_RATE_KTO = 2e-5
LEARNING_RATE_ORPO = 3e-5
LEARNING_RATE_GRPO = 2.5e-5

# Significantly increased epochs for deeper learning
NUM_EPOCHS_SFT = 20  # Increased from 15
NUM_EPOCHS_KTO = 15  # Increased from 10
NUM_EPOCHS_ORPO = 18  # Increased from 12
NUM_EPOCHS_GRPO = 8  # Increased from 5

WARMUP_RATIO = 0.2
LOGGING_STEPS = 5

device = "cuda" if torch.cuda.is_available() else "cpu"

# Simple test questions for consistent evaluation
TEST_QUESTIONS = [
    "How do I cook pasta perfectly?",
    "What's the best way to scramble eggs?",
    "How do I make rice that isn't sticky?",
    "What's an easy dinner for beginners?",
    "How do I know when chicken is cooked?",
]


def install_packages():
    packages = [
        "torch>=2.0.0",
        "transformers>=4.36.0",
        "trl>=0.7.4",
        "datasets>=2.14.0",
        "accelerate>=0.21.0",
    ]
    for pkg in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        except:
            pass


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def monitor_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(device) / (1024**3)
        reserved = torch.cuda.memory_reserved(device) / (1024**3)
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
        return reserved < 7.5
    return True


def test_model(model, tokenizer, prompt):
    model.eval()
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH
    )
    if torch.cuda.is_available():
        inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
            use_cache=False,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    return response.strip()


def evaluate_stage(model, tokenizer, stage_name):
    print(f"\n{'='*60}")
    print(f"{stage_name.upper()} MODEL EVALUATION")
    print(f"{'='*60}")

    for i, question in enumerate(TEST_QUESTIONS, 1):
        print(f"\nQ{i}: {question}")
        print("-" * 40)
        response = test_model(model, tokenizer, question)
        print(f"Answer: {response}")


install_packages()

In [2]:
# =============================================================================
# BASE MODEL LOADING AND EVALUATION
# =============================================================================

print("=" * 80)
print("LOADING BASE MODEL FOR INITIAL EVALUATION")
print("=" * 80)

# Load model with fixed configuration
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)

# Configure tokenizer with proper settings
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Resize token embeddings after adding special tokens
model.resize_token_embeddings(len(tokenizer))

monitor_memory()
evaluate_stage(model, tokenizer, "BASE")

LOADING BASE MODEL FOR INITIAL EVALUATION
GPU Memory: 0.92GB allocated, 1.19GB reserved

BASE MODEL EVALUATION

Q1: How do I cook pasta perfectly?
----------------------------------------
Answer: Cooking pasta to perfection can be a bit tricky, but with the right techniques and ingredients, you'll be able to achieve the perfect texture and flavor. Here's a step-by-step guide on how to cook pasta:

1. Choose your pasta: Select a type of pasta that suits your taste preferences. Popular choices include spaghetti, penne, fusilli, and ravioli.

2. Measure the water: For all types of pasta, measure about 4 cups of boiling water per pound of pasta dough.

3. Cook the pasta: Place the pasta in a large pot and cover it with cold water. Bring the water to a boil, then reduce the heat to low and simmer for about 8-10 minutes or until al dente.

4. Drain and rinse: Once the pasta is cooked, drain it under cold running water and pat it dry with paper towels.

5. Prepare sauce: In a small bowl, whis

# Supervised fine-tuning (SFT)

**Mathematical Foundation:**
```
L_SFT = -∑(t=1 to T) log P(y_t | x, y_<t; θ)
```

**Process:**
1. Model learns to predict next tokens given context
2. Cross-entropy loss measures prediction accuracy
3. Backpropagation updates model parameters
4. Domain-specific examples adapt model behavior

**Objective:** Transform general model into cooking specialist

In [3]:
# =============================================================================
# SUPERVISED FINE-TUNING (SFT)
# =============================================================================


def create_expanded_sft_dataset():
    """Create comprehensive SFT dataset for cooking domain"""

    examples = [
        {
            "text": "How do I cook pasta perfectly?\n\nCooking perfect pasta requires attention to water, salt, and timing.\n\n**Water Setup:**\nUse a large pot with plenty of water - about 4-6 quarts for 1 pound of pasta. The pasta needs room to move freely without sticking together.\n\n**Salt the Water:**\nAdd 1-2 tablespoons of salt when the water reaches a rolling boil. The water should taste like mild seawater. This is your only chance to season the pasta itself.\n\n**Cooking Process:**\nAdd pasta to boiling water and stir immediately to prevent sticking. Cook for 1-2 minutes less than package directions for al dente texture.\n\n**Testing Doneness:**\nTaste test frequently in the final minutes. Perfect pasta should have a slight firmness when you bite it - tender but not mushy.\n\n**Finishing:**\nReserve 1 cup of pasta water before draining. This starchy water helps bind sauces to pasta. Never rinse pasta unless making a cold salad."
        },
        {
            "text": "What's the best way to scramble eggs?\n\nGreat scrambled eggs are all about gentle heat and constant movement.\n\n**Preparation:**\nCrack 2-3 eggs per person into a bowl. Add a pinch of salt and 1 tablespoon of butter or cream per 2 eggs. Whisk until completely smooth with no streaks.\n\n**Heat Control:**\nUse medium-low heat in a non-stick pan. Add a small amount of butter and let it melt completely without browning.\n\n**Cooking Technique:**\nPour in eggs and let them sit for 20-30 seconds. Using a spatula, gently push cooked edges toward center, tilting pan to let raw egg flow underneath.\n\n**The Key:**\nContinuous gentle movement is crucial. Keep pushing and folding, never scrambling vigorously. Remove from heat when eggs still look slightly wet - they'll finish cooking from residual heat.\n\n**Serving:**\nServe immediately on warmed plates. Season with fresh pepper and herbs if desired."
        },
        {
            "text": "How do I make rice that isn't sticky?\n\nPerfect rice comes down to the right ratio, technique, and not lifting the lid.\n\n**Rice-to-Water Ratio:**\nFor long-grain white rice, use 1 cup rice to 1.5 cups water. For brown rice, use 1 cup rice to 2 cups water.\n\n**Rinsing:**\nRinse rice in cold water until water runs clear - usually 3-4 rinses. This removes excess starch that causes stickiness.\n\n**Cooking Method:**\nCombine rice and water in heavy-bottomed pot. Bring to a boil, then immediately reduce to lowest heat setting. Cover tightly with lid.\n\n**Timing:**\nWhite rice: 18 minutes. Brown rice: 45 minutes. Never lift the lid during cooking - this releases steam needed for proper cooking.\n\n**Resting:**\nAfter cooking time, remove from heat but keep lid on for 10 minutes. Then fluff gently with fork, not spoon.\n\n**Pro Tip:**\nFor extra fluffy rice, add 1 teaspoon of butter or oil to the water before cooking."
        },
        {
            "text": "What's an easy dinner for beginners?\n\nOne-pan chicken and vegetables is perfect for beginners - minimal cleanup and hard to mess up.\n\n**Ingredients:**\n4 chicken thighs, 2 cups baby potatoes (halved), 1 cup carrots (chopped), 1 onion (sliced), olive oil, salt, pepper, dried herbs.\n\n**Preparation:**\nPreheat oven to 425°F. Pat chicken dry and season generously with salt and pepper. Cut vegetables into similar-sized pieces so they cook evenly.\n\n**Assembly:**\nToss vegetables with 2 tablespoons olive oil, salt, and pepper. Spread on large baking sheet. Place seasoned chicken on top, skin side up.\n\n**Cooking:**\nBake for 35-40 minutes until chicken reaches 165°F internal temperature and vegetables are tender when pierced with fork.\n\n**Serving:**\nLet rest 5 minutes before serving. The chicken juices flavor the vegetables beautifully.\n\n**Variations:**\nTry different vegetable combinations - broccoli, bell peppers, zucchini all work well."
        },
        {
            "text": "How do I know when chicken is cooked?\n\nFood safety is crucial with chicken - use multiple indicators to ensure doneness.\n\n**Internal Temperature:**\nUse instant-read thermometer in thickest part of meat, not touching bone. Chicken must reach 165°F (74°C) throughout.\n\n**Visual Cues:**\nProperly cooked chicken has no pink color in the meat. Juices should run clear, not pink or red, when pierced with knife.\n\n**Texture Test:**\nCooked chicken feels firm to touch, not soft or squishy. Raw chicken has a distinctly different, softer texture.\n\n**Cooking Times:**\nBoneless breasts: 6-8 minutes per side. Bone-in thighs: 25-30 minutes total. Whole chicken: 20 minutes per pound at 375°F.\n\n**Rest Time:**\nLet chicken rest 5-10 minutes after cooking. Internal temperature will continue rising 5-10 degrees, ensuring safety.\n\n**Safety Note:**\nWhen in doubt, cook longer. Overcooked chicken is better than foodborne illness. Clean all surfaces that touched raw chicken."
        },
        {
            "text": "How do I make a simple tomato sauce?\n\nHomemade tomato sauce is easier than you think and tastes much better than jarred.\n\n**Base Ingredients:**\n1 can (28 oz) crushed tomatoes, 3 cloves garlic (minced), 1/4 cup olive oil, 1 medium onion (diced), salt, pepper, dried basil.\n\n**Building Flavor:**\nHeat olive oil in large pan over medium heat. Add diced onion and cook until translucent, about 5 minutes. Add minced garlic and cook 30 seconds until fragrant.\n\n**Adding Tomatoes:**\nPour in crushed tomatoes, add 1 teaspoon salt, 1/2 teaspoon pepper, and 1 teaspoon dried basil. Stir to combine.\n\n**Simmering:**\nBring to gentle boil, then reduce heat to low. Simmer uncovered for 20-30 minutes, stirring occasionally, until sauce thickens to desired consistency.\n\n**Finishing:**\nTaste and adjust seasoning. Add fresh basil leaves in final 5 minutes if available.\n\n**Storage:**\nCool completely before refrigerating up to 1 week or freezing up to 3 months."
        },
        {
            "text": "What's the secret to crispy bacon?\n\nCrispy bacon requires patience and the right technique - no flipping frantically required.\n\n**Cold Start Method:**\nPlace bacon strips in cold pan without overlapping. Turn heat to medium-low and cook slowly, allowing fat to render gradually.\n\n**The Patient Approach:**\nDon't move bacon for first 3-4 minutes. Let fat render out slowly - this is what creates crispiness without burning.\n\n**Gentle Flipping:**\nWhen edges start curling, flip once and continue cooking until desired crispiness. Total time: 8-12 minutes depending on thickness.\n\n**Oven Method:**\nAlternatively, arrange on rimmed baking sheet and bake at 400°F for 15-20 minutes. No flipping needed and less splatter.\n\n**Draining:**\nTransfer to paper towel-lined plate immediately. Don't let bacon sit in its own fat or it will become soggy.\n\n**Save the Fat:**\nStrain and save bacon fat for cooking potatoes, vegetables, or cornbread - it adds incredible flavor."
        },
        {
            "text": "How do I boil eggs perfectly?\n\nPerfect boiled eggs depend on timing and temperature control.\n\n**Starting Setup:**\nPlace eggs in single layer in saucepan. Cover with cold water by 1 inch. This ensures even cooking.\n\n**Heating Process:**\nBring water to rolling boil over high heat. Once boiling, remove from heat and cover pot immediately.\n\n**Timing for Results:**\nSoft-boiled (runny yolk): 4-6 minutes. Medium-boiled (jammy yolk): 7-9 minutes. Hard-boiled (firm yolk): 10-12 minutes.\n\n**Ice Bath Stop:**\nImmediately transfer eggs to ice water bath to stop cooking process. This prevents overcooking and green ring around yolk.\n\n**Peeling Technique:**\nFor easier peeling, use week-old eggs rather than fresh. Crack shell all over and peel under cool running water.\n\n**Pro Tip:**\nAdd 1 teaspoon baking soda to boiling water for easier-to-peel eggs."
        },
        {
            "text": "What's the best way to cook steak?\n\nGreat steak requires high heat, proper seasoning, and knowing when to stop.\n\n**Steak Selection:**\nChoose steaks at least 1 inch thick - ribeye, strip, or filet work well. Let come to room temperature 30 minutes before cooking.\n\n**Seasoning:**\nSeason generously with coarse salt and black pepper 40 minutes before cooking, or right before if short on time.\n\n**High Heat Cooking:**\nHeat cast iron pan or grill to high heat. Add small amount of oil with high smoke point (not olive oil).\n\n**Cooking Process:**\nSear steak 3-4 minutes per side for medium-rare, depending on thickness. Don't move or press down while cooking.\n\n**Temperature Guide:**\nRare: 120-125°F, Medium-rare: 130-135°F, Medium: 135-145°F. Use instant-read thermometer for accuracy.\n\n**Resting:**\nLet steak rest 5-10 minutes before slicing. This redistributes juices for better flavor and texture."
        },
        {
            "text": "How do I make fluffy pancakes?\n\nFluffy pancakes require gentle mixing and proper heat control.\n\n**Dry Ingredients:**\nWhisk together 2 cups flour, 2 tablespoons sugar, 2 teaspoons baking powder, and 1/2 teaspoon salt in large bowl.\n\n**Wet Ingredients:**\nIn separate bowl, whisk 1 3/4 cups milk, 2 eggs, and 1/4 cup melted butter until combined.\n\n**Critical Mixing:**\nPour wet ingredients into dry ingredients. Stir just until barely combined - lumps are okay! Overmixing creates tough, flat pancakes.\n\n**Pan Preparation:**\nHeat non-stick pan or griddle over medium heat. Lightly grease with butter or oil.\n\n**Cooking:**\nPour 1/4 cup batter per pancake. Cook until bubbles form on surface and edges look set, about 2-3 minutes. Flip once and cook 1-2 minutes more.\n\n**Serving:**\nServe immediately while hot, or keep warm in 200°F oven until ready to serve."
        },
        {
            "text": "What's a simple salad dressing recipe?\n\nBasic vinaigrette is the foundation of great salads and takes just minutes to make.\n\n**Classic Ratio:**\n3 parts oil to 1 part acid (vinegar or lemon juice). For example: 3 tablespoons olive oil to 1 tablespoon vinegar.\n\n**Building Flavor:**\nAdd 1 teaspoon Dijon mustard for emulsification and flavor. Season with salt and pepper to taste.\n\n**Mixing Method:**\nWhisk acid, mustard, salt, and pepper in small bowl. Slowly drizzle in oil while whisking constantly to create emulsion.\n\n**Variations:**\nBalsamic vinegar for sweet flavor, red wine vinegar for classic taste, lemon juice for brightness. Add minced garlic or herbs for extra flavor.\n\n**Storage:**\nStore in sealed jar in refrigerator up to 1 week. Shake well before using as ingredients will separate.\n\n**Pro Tip:**\nMake larger batches in mason jars - just shake before each use."
        },
        {
            "text": "How do I roast vegetables properly?\n\nRoasting brings out natural sweetness in vegetables through caramelization.\n\n**Preparation:**\nPreheat oven to 425°F. Cut vegetables into uniform pieces so they cook evenly - about 1-inch pieces work well.\n\n**Oil and Seasoning:**\nToss vegetables with olive oil (about 1-2 tablespoons per baking sheet), salt, and pepper. Don't oversaturate with oil.\n\n**Pan Setup:**\nSpread vegetables in single layer on baking sheet. Overcrowding creates steam instead of roasting, preventing caramelization.\n\n**Roasting Time:**\nMost vegetables take 20-30 minutes. Harder vegetables like potatoes need longer; softer ones like zucchini need less time.\n\n**Testing Doneness:**\nVegetables should be tender when pierced with fork and have golden-brown edges. This browning is where flavor develops.\n\n**Finishing:**\nSeason with fresh herbs, lemon juice, or parmesan cheese while vegetables are still hot."
        },
    ]

    return examples


def run_enhanced_sft_training(model, tokenizer):
    print("=" * 80)
    print("SUPERVISED FINE-TUNING (SFT) - ENHANCED TRAINING")
    print("=" * 80)
    print("Mathematical Foundation: L_SFT = -∑ log P(y_t | x, y_<t; θ)")
    print(f"Training: {NUM_EPOCHS_SFT} epochs, LR: {LEARNING_RATE_SFT}")

    sft_data = create_expanded_sft_dataset()
    dataset = Dataset.from_list(sft_data)

    print(f"Dataset: {len(sft_data)} comprehensive cooking examples")
    print(f"Effective batch size: {BATCH_SIZE_SFT * GRAD_ACCUM_SFT}")

    try:
        from trl import SFTTrainer, SFTConfig

        config = SFTConfig(
            output_dir="./temp_sft",
            num_train_epochs=NUM_EPOCHS_SFT,
            per_device_train_batch_size=BATCH_SIZE_SFT,
            gradient_accumulation_steps=GRAD_ACCUM_SFT,
            learning_rate=LEARNING_RATE_SFT,
            max_length=MAX_LENGTH,
            logging_steps=LOGGING_STEPS,
            save_strategy="no",
            fp16=False,
            bf16=torch.cuda.is_available(),
            warmup_ratio=WARMUP_RATIO,
            remove_unused_columns=False,
            dataset_text_field="text",
            gradient_checkpointing=True,
            dataloader_drop_last=True,
        )

        trainer = SFTTrainer(
            model=model,
            args=config,
            train_dataset=dataset,
            processing_class=tokenizer,
        )

        trainer.train()
        del trainer

    except Exception as e:
        print(f"SFT training error: {e}")

    cleanup_memory()
    return model, tokenizer


# Execute Enhanced SFT
model, tokenizer = run_enhanced_sft_training(model, tokenizer)
evaluate_stage(model, tokenizer, "SFT")

SUPERVISED FINE-TUNING (SFT) - ENHANCED TRAINING
Mathematical Foundation: L_SFT = -∑ log P(y_t | x, y_<t; θ)
Training: 20 epochs, LR: 4e-05
Dataset: 12 comprehensive cooking examples
Effective batch size: 16


Adding EOS to train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,1.761900
10,0.232200
15,0.014300
20,0.010400



SFT MODEL EVALUATION

Q1: How do I cook pasta perfectly?
----------------------------------------
Answer: Perfect pasta requires attention to water, salt, and timing.

**Water Setup:**
Use a large pot with plenty of water - about 4-6 quarts for 1 pound of pasta. The pasta needs room to move freely without sticking together.

**Salt the Water:**
Add 1-2 tablespoons of salt when the water reaches a rolling boil. The water should taste like mild seawater. This is your only chance to season the pasta itself.

**Cooking Process:**
Add pasta to boiling water and stir immediately to prevent sticking. Cook for 1-2 minutes less than package directions for al dente texture.

**Testing Doneness:**
Taste test frequently in the final minutes. Perfect pasta should have a slight firmness when you bite it - tender but not mushy.

**Finishing:**
Reserve 1 cup of pasta water before draining. This starchy water helps bind sauces to pasta. Never rinse pasta unless making a cold salad.

Q2: What's the bes

# Kahneman-Tversky Optimization (KTO)

## Kahneman-Tversky Optimization Theory

**Foundation:** Nobel Prize-winning Prospect Theory

**Value Function:**
```
v(r) = {
  r^α           if r ≥ 0  (gains)
  -λ(-r)^β      if r < 0  (losses)
}
```

**Key Insights:**
- λ > 1: Loss aversion (losses feel worse than equivalent gains)
- α, β < 1: Diminishing sensitivity to magnitude changes
- Reference-dependent preferences (not absolute values)

**Training:** Model learns human preference psychology patterns

In [4]:
# =============================================================================
# KAHNEMAN-TVERSKY OPTIMIZATION (KTO) - BEHAVIORAL ECONOMICS
# =============================================================================


def create_expanded_kto_dataset():
    """Create comprehensive KTO dataset with clear preference patterns"""

    examples = [
        # High-quality responses (label=True)
        {
            "prompt": "How do I cook pasta perfectly?",
            "completion": "Perfect pasta requires three key elements: plenty of salted boiling water, proper timing, and pasta water for finishing.\n\nUse a large pot with 4-6 quarts of water per pound of pasta. When it reaches a rolling boil, add 1-2 tablespoons of salt - the water should taste like mild seawater.\n\nAdd pasta and stir immediately to prevent sticking. Cook for 1-2 minutes less than package directions for al dente texture. Taste test frequently in the final minutes.\n\nBefore draining, reserve 1 cup of pasta water. This starchy water helps bind sauces. Never rinse pasta unless making a cold salad.",
            "label": True,
        },
        {
            "prompt": "What's the best way to scramble eggs?",
            "completion": "The secret to creamy scrambled eggs is low heat and constant gentle movement.\n\nWhisk 2-3 eggs with a pinch of salt and 1 tablespoon of butter or cream. Use medium-low heat in a non-stick pan with melted butter.\n\nPour in eggs and let sit for 20-30 seconds. Gently push cooked edges toward center, tilting pan to let raw egg flow underneath. Keep moving gently - never scramble vigorously.\n\nRemove from heat while eggs still look slightly wet. They'll finish cooking from residual heat. Serve immediately on warmed plates.",
            "label": True,
        },
        {
            "prompt": "How do I make rice that isn't sticky?",
            "completion": "Non-sticky rice comes from proper rinsing, correct ratios, and never lifting the lid during cooking.\n\nRinse 1 cup rice in cold water until water runs clear (3-4 rinses). This removes excess starch that causes stickiness.\n\nUse 1 cup rice to 1.5 cups water for white rice. Bring to boil, then immediately reduce to lowest heat. Cover tightly and cook 18 minutes without lifting lid.\n\nAfter cooking, remove from heat but keep covered for 10 more minutes. Then fluff gently with a fork, not a spoon.",
            "label": True,
        },
        # Lower-quality responses (label=False)
        {
            "prompt": "How do I cook pasta perfectly?",
            "completion": "Boil pasta in water until it's soft. Add salt if you want. Drain when done.",
            "label": False,
        },
        {
            "prompt": "What's the best way to scramble eggs?",
            "completion": "Beat eggs and cook them in a pan. Stir them around until they're cooked.",
            "label": False,
        },
        {
            "prompt": "How do I make rice that isn't sticky?",
            "completion": "Use less water and don't overcook it. Rinse the rice first.",
            "label": False,
        },
        # Additional preference pairs for deeper learning
        {
            "prompt": "What's an easy dinner for beginners?",
            "completion": "One-pan chicken and vegetables is perfect for beginners - minimal cleanup and foolproof results.\n\nPreheat oven to 425°F. Season 4 chicken thighs with salt and pepper. Toss halved baby potatoes, chopped carrots, and sliced onion with olive oil, salt, and pepper.\n\nSpread vegetables on large baking sheet, place chicken on top skin-side up. Bake 35-40 minutes until chicken reaches 165°F internal temperature and vegetables are fork-tender.\n\nLet rest 5 minutes before serving. The chicken juices naturally flavor the vegetables.",
            "label": True,
        },
        {
            "prompt": "What's an easy dinner for beginners?",
            "completion": "Cook some chicken and vegetables. Put them in the oven until done.",
            "label": False,
        },
        {
            "prompt": "How do I know when chicken is cooked?",
            "completion": "Food safety is crucial with chicken - use multiple indicators to ensure proper doneness.\n\nUse an instant-read thermometer in the thickest part, avoiding bone. Chicken must reach 165°F internal temperature throughout.\n\nVisually, cooked chicken has no pink color and juices run clear when pierced. The texture should feel firm, not soft or squishy.\n\nWhen in doubt, cook longer. Let chicken rest 5-10 minutes after cooking - internal temperature continues rising.",
            "label": True,
        },
        {
            "prompt": "How do I know when chicken is cooked?",
            "completion": "Cook it until it looks done and isn't pink inside.",
            "label": False,
        },
        {
            "prompt": "How do I make fluffy pancakes?",
            "completion": "Fluffy pancakes require gentle mixing and proper heat control - overmixing is the enemy.\n\nWhisk dry ingredients (2 cups flour, 2 tbsp sugar, 2 tsp baking powder, 1/2 tsp salt) in one bowl. Mix wet ingredients (1 3/4 cups milk, 2 eggs, 1/4 cup melted butter) in another.\n\nPour wet into dry ingredients and stir just until barely combined - lumps are perfectly fine! Overmixing develops gluten, creating tough pancakes.\n\nCook on medium heat, flipping when bubbles form on surface and edges look set.",
            "label": True,
        },
        {
            "prompt": "How do I make fluffy pancakes?",
            "completion": "Mix pancake batter and cook on a pan until both sides are done.",
            "label": False,
        },
        {
            "prompt": "What's the secret to crispy bacon?",
            "completion": "Crispy bacon requires patience and gradual fat rendering - start cold and go slow.\n\nPlace bacon in cold pan without overlapping. Turn heat to medium-low and cook slowly, allowing fat to render gradually over 3-4 minutes before first flip.\n\nThis slow rendering creates crispiness without burning. Total cooking time is 8-12 minutes depending on thickness.\n\nAlternatively, bake at 400°F on rimmed baking sheet for 15-20 minutes - no flipping needed and less splatter.",
            "label": True,
        },
        {
            "prompt": "What's the secret to crispy bacon?",
            "completion": "Cook bacon on high heat and flip it a lot until crispy.",
            "label": False,
        },
        {
            "prompt": "How do I boil eggs perfectly?",
            "completion": "Perfect boiled eggs depend on precise timing and immediate temperature control.\n\nPlace eggs in single layer, cover with cold water by 1 inch. Bring to rolling boil, then immediately remove from heat and cover.\n\nTiming: Soft-boiled 4-6 minutes, medium 7-9 minutes, hard-boiled 10-12 minutes. Immediately transfer to ice water to stop cooking.\n\nFor easier peeling, use week-old eggs and crack shell all over before peeling under running water.",
            "label": True,
        },
        {
            "prompt": "How do I boil eggs perfectly?",
            "completion": "Put eggs in hot water and cook until done.",
            "label": False,
        },
    ]

    return examples


def run_enhanced_kto_training(model, tokenizer):
    print("=" * 80)
    print("KAHNEMAN-TVERSKY OPTIMIZATION (KTO) - BEHAVIORAL ECONOMICS")
    print("=" * 80)
    print("Theory: v(r) = r^α (gains) vs -λ(-r)^β (losses)")
    print(f"Training: {NUM_EPOCHS_KTO} epochs, LR: {LEARNING_RATE_KTO}")

    cleanup_memory()

    kto_data = create_expanded_kto_dataset()
    dataset = Dataset.from_list(kto_data)

    print(f"Dataset: {len(kto_data)} preference pairs")
    print(f"Effective batch size: {BATCH_SIZE_KTO * GRAD_ACCUM_KTO}")

    try:
        from trl import KTOTrainer, KTOConfig

        config = KTOConfig(
            output_dir="./temp_kto",
            num_train_epochs=NUM_EPOCHS_KTO,
            per_device_train_batch_size=BATCH_SIZE_KTO,
            gradient_accumulation_steps=GRAD_ACCUM_KTO,
            learning_rate=LEARNING_RATE_KTO,
            max_length=MAX_LENGTH,
            max_prompt_length=MAX_LENGTH // 2,
            logging_steps=LOGGING_STEPS,
            save_strategy="no",
            fp16=False,
            bf16=torch.cuda.is_available(),
            warmup_ratio=WARMUP_RATIO,
            remove_unused_columns=False,
            gradient_checkpointing=True,
            dataloader_drop_last=True,
        )

        trainer = KTOTrainer(
            model=model,
            args=config,
            train_dataset=dataset,
            processing_class=tokenizer,
        )

        trainer.train()
        del trainer

    except Exception as e:
        print(f"KTO training error: {e}")

    cleanup_memory()
    return model, tokenizer


# Execute Enhanced KTO
model, tokenizer = run_enhanced_kto_training(model, tokenizer)
evaluate_stage(model, tokenizer, "KTO")

KAHNEMAN-TVERSKY OPTIMIZATION (KTO) - BEHAVIORAL ECONOMICS
Theory: v(r) = r^α (gains) vs -λ(-r)^β (losses)
Training: 15 epochs, LR: 2e-05
Dataset: 16 preference pairs
Effective batch size: 16


Extracting prompt from train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Processing tokenized train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Extracting KL train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Processing tokenized train KL dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
5,0.261900
10,0.041800
15,0.066800



KTO MODEL EVALUATION

Q1: How do I cook pasta perfectly?
----------------------------------------
Answer: Perfect pasta requires three key elements: plenty of salted boiling water, proper timing, and pasta water for finishing.

Use a large pot with 4-6 quarts of water per pound of pasta. When it reaches a rolling boil, add 1-2 tablespoons of salt - the water should taste like mild seawater.

Add pasta and stir immediately to prevent sticking. Cook for 1-2 minutes less than package directions for al dente texture. Taste test frequently in the final minutes.

Before draining, reserve 1 cup of pasta water. This starchy water helps bind sauces. Never rinse pasta unless making a cold salad.

Q2: What's the best way to scramble eggs?
----------------------------------------
Answer: The secret to creamy scrambled eggs is low heat and constant gentle movement.

Whisk 2-3 eggs with a pinch of salt and 1 tablespoon of butter or cream. Use medium-low heat in a non-stick pan with melted butter.



# Odds Ratio Preference Optimization (ORPO)

## Odds Ratio Preference Optimization Theory

**Innovation:** Combines SFT and preference learning in single step

**Loss Function:**
```
L_ORPO = L_SFT + λ * L_OR
```

**Odds Ratio Loss:**
```
L_OR = -log(σ(log(P(y_chosen|x) / P(y_rejected|x))))
```

**Advantages:**
- No separate reference model needed (memory efficient)
- Single training phase (time efficient)
- Direct optimization of preferences during SFT

In [5]:
# =============================================================================
# ODDS RATIO PREFERENCE OPTIMIZATION (ORPO)
# =============================================================================


def create_expanded_orpo_dataset():
    """Create comprehensive ORPO preference pairs with strong contrasts"""

    examples = [
        {
            "prompt": "How do I make a simple tomato sauce?",
            "chosen": "Homemade tomato sauce is surprisingly simple and far superior to store-bought versions.\n\nHeat 1/4 cup olive oil in large pan over medium heat. Add 1 diced onion and cook until translucent, about 5 minutes. Add 3 minced garlic cloves and cook 30 seconds until fragrant.\n\nPour in 1 can (28 oz) crushed tomatoes, add 1 tsp salt, 1/2 tsp pepper, and 1 tsp dried basil. Bring to gentle boil, then reduce heat to low.\n\nSimmer uncovered 20-30 minutes, stirring occasionally, until sauce thickens to your liking. Taste and adjust seasoning. Add fresh basil in final 5 minutes if available.",
            "rejected": "Just heat up some tomatoes with garlic and onion until it looks like sauce.",
        },
        {
            "prompt": "What's the best way to cook steak?",
            "chosen": "Perfect steak requires high heat, proper seasoning, and precise timing.\n\nChoose steaks at least 1 inch thick and let them reach room temperature for 30 minutes. Season generously with coarse salt and black pepper.\n\nHeat cast iron pan or grill to high heat. Add small amount of high-smoke-point oil (not olive oil). Sear steak 3-4 minutes per side for medium-rare, depending on thickness.\n\nUse instant-read thermometer: 130-135°F for medium-rare. Let steak rest 5-10 minutes before slicing to redistribute juices.",
            "rejected": "Cook steak on high heat until it looks done. Season with salt and pepper.",
        },
        {
            "prompt": "How do I roast vegetables properly?",
            "chosen": "Proper roasting brings out vegetables' natural sweetness through caramelization.\n\nPreheat oven to 425°F. Cut vegetables into uniform 1-inch pieces for even cooking. Toss with olive oil (1-2 tbsp per baking sheet), salt, and pepper.\n\nSpread in single layer on baking sheet - overcrowding creates steam instead of roasting. Most vegetables take 20-30 minutes.\n\nVegetables are done when fork-tender with golden-brown edges. This browning is crucial for flavor development. Finish with fresh herbs or lemon juice while hot.",
            "rejected": "Put vegetables in the oven with oil until they're soft.",
        },
        {
            "prompt": "How do I make perfect mashed potatoes?",
            "chosen": "Creamy mashed potatoes require the right potato variety and proper technique.\n\nUse starchy potatoes like Russets or Yukon Gold. Peel and cut into uniform chunks. Start in cold, salted water and bring to boil - this ensures even cooking.\n\nCook until fork-tender, about 15-20 minutes. Drain thoroughly and let sit 2-3 minutes to evaporate excess moisture.\n\nMash while hot using potato masher or ricer. Gradually add warm butter and milk/cream, starting with 4 tbsp butter and 1/2 cup liquid per 2 lbs potatoes. Season with salt and white pepper.",
            "rejected": "Boil potatoes and mash them with butter and milk.",
        },
        {
            "prompt": "What's the secret to good fried rice?",
            "chosen": "Great fried rice uses day-old rice and high heat for proper texture and flavor.\n\nUse cold, day-old rice - fresh rice is too moist and creates mushy results. Break up any clumps with your hands before cooking.\n\nHeat wok or large pan over high heat. Add oil and scrambled eggs first, remove and set aside. Add aromatics (garlic, ginger) for 30 seconds.\n\nAdd cold rice, breaking up clumps as you stir-fry for 3-4 minutes. Add soy sauce, vegetables, and cooked eggs back in. Keep everything moving over high heat for best texture.",
            "rejected": "Fry rice with soy sauce and whatever vegetables you have.",
        },
        {
            "prompt": "How do I make bread rise properly?",
            "chosen": "Proper bread rising depends on yeast activation, temperature control, and timing.\n\nProof yeast in warm water (105-110°F) with a pinch of sugar for 5-10 minutes until foamy. Water too hot kills yeast, too cold won't activate it.\n\nFirst rise should happen in oiled bowl, covered, in warm spot (75-80°F) until doubled in size - usually 1-2 hours. Punch down gently and shape.\n\nSecond rise is shorter, 30-60 minutes until puffy but not necessarily doubled. Test with gentle finger poke - dough should spring back slowly when ready to bake.",
            "rejected": "Let dough sit in a warm place until it gets bigger.",
        },
        {
            "prompt": "What makes cookies chewy vs crispy?",
            "chosen": "Cookie texture depends on ingredient ratios, baking time, and temperature control.\n\nFor chewy cookies: Use more brown sugar than white (brown sugar's molasses adds moisture), slightly underbake, and use bread flour or add extra egg yolk for chewiness.\n\nFor crispy cookies: Use more white sugar, butter at room temperature, bake longer until edges are golden, and use all-purpose flour.\n\nBaking temperature also matters: lower temperature (325°F) spreads cookies more and creates chewier texture, higher temperature (375°F) sets edges quickly for crispier results.",
            "rejected": "Different ingredients and baking times make cookies different textures.",
        },
        {
            "prompt": "How do I keep salad greens fresh longer?",
            "chosen": "Proper storage dramatically extends salad green freshness.\n\nWash greens in cold water, then dry thoroughly using salad spinner or paper towels - excess moisture causes quick spoilage.\n\nStore in refrigerator in breathable container or plastic bag with paper towels to absorb moisture. Don't seal completely airtight - greens need some air circulation.\n\nFor delicate greens like arugula: use within 3-5 days. Hardier greens like romaine: 7-10 days. Remove any yellowing or slimy leaves immediately to prevent spread.",
            "rejected": "Keep salad in the fridge in a bag.",
        },
    ]

    return examples


def run_enhanced_orpo_training(model, tokenizer):
    print("=" * 80)
    print("ODDS RATIO PREFERENCE OPTIMIZATION (ORPO) - MONOLITHIC TRAINING")
    print("=" * 80)
    print("Mathematical Foundation: L_ORPO = L_SFT + λ * L_OR")
    print(f"Training: {NUM_EPOCHS_ORPO} epochs, LR: {LEARNING_RATE_ORPO}")

    cleanup_memory()

    orpo_data = create_expanded_orpo_dataset()
    dataset = Dataset.from_list(orpo_data)

    print(f"Dataset: {len(orpo_data)} preference pairs")
    print(f"Effective batch size: {BATCH_SIZE_ORPO * GRAD_ACCUM_ORPO}")

    try:
        from trl import ORPOTrainer, ORPOConfig

        config = ORPOConfig(
            output_dir="./temp_orpo",
            num_train_epochs=NUM_EPOCHS_ORPO,
            per_device_train_batch_size=BATCH_SIZE_ORPO,
            gradient_accumulation_steps=GRAD_ACCUM_ORPO,
            learning_rate=LEARNING_RATE_ORPO,
            max_length=MAX_LENGTH,
            max_prompt_length=MAX_LENGTH // 2,
            logging_steps=LOGGING_STEPS,
            save_strategy="no",
            fp16=False,
            bf16=torch.cuda.is_available(),
            warmup_ratio=WARMUP_RATIO,
            remove_unused_columns=False,
            gradient_checkpointing=True,
            dataloader_drop_last=True,
        )

        trainer = ORPOTrainer(
            model=model,
            args=config,
            train_dataset=dataset,
            processing_class=tokenizer,
        )

        trainer.train()
        del trainer

    except Exception as e:
        print(f"ORPO training error: {e}")

    cleanup_memory()
    return model, tokenizer


# Execute Enhanced ORPO
model, tokenizer = run_enhanced_orpo_training(model, tokenizer)
evaluate_stage(model, tokenizer, "ORPO")

ODDS RATIO PREFERENCE OPTIMIZATION (ORPO) - MONOLITHIC TRAINING
Mathematical Foundation: L_ORPO = L_SFT + λ * L_OR
Training: 18 epochs, LR: 3e-05
Dataset: 8 preference pairs
Effective batch size: 12


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Step,Training Loss
5,2.117100
10,0.484900
15,0.057300



ORPO MODEL EVALUATION

Q1: How do I cook pasta perfectly?
----------------------------------------
Answer: Perfect cooked pasta requires the right timing, water, and type of pasta.

Use a large pot with plenty of water (about 4-6 quarts for 1 pound of pasta). Bring to gentle boil, then reduce heat to low.

Cook until al dente - tender but not mushy. Taste and adjust seasoning. Add fresh basil or salt water while hot for better texture.

For al dente: Taste test frequently in the final minutes. Pasta should have a slight firmness when you bite it - this indicates proper cooking.

Q2: What's the best way to scramble eggs?
----------------------------------------
Answer: Creamy scrambled eggs are all about high heat and proper seasoning.

Heat a non-stick pan over high heat. Add a pinch of salt and 2 tbsp. oil. Crack 2-3 eggs and gently scramble, being careful not to scramble vigorously.

Pour in seasonings - fresh herbs, lemon juice, or tomato sauce (if using). Let it simmer for 20-30 s

# Group Relative Policy Optimization (GRPO)

## Group Relative Policy Optimization Theory

**Innovation:** Eliminates value function using group mean baseline

**Traditional PPO:** A(s,a) = r(s,a) - V_φ(s)
**GRPO Advantage:** A_i = r_i - (1/N) ∑(j=1 to N) r_j

**Algorithm:**
1. Generate N responses from current policy
2. Collect human preference ratings
3. Calculate group mean as baseline
4. Compute advantages relative to group performance
5. Update policy using advantage-weighted gradients

**Benefits:** 50% memory reduction vs PPO, direct human feedback

In [6]:
# =============================================================================
# GROUP RELATIVE POLICY OPTIMIZATION (GRPO)
# =============================================================================


def create_grpo_scenarios():
    return [
        "How do I cook vegetables without making them mushy?",
        "What's the easiest way to improve my cooking?",
        "How do I know when meat is properly cooked?",
    ]


def interactive_feedback(model, tokenizer, prompt):
    print(f"\n{'='*60}")
    print(f"SCENARIO: {prompt}")
    print(f"{'='*60}")

    # Generate 3 diverse responses
    responses = []
    print("Generating 3 responses for your evaluation...\n")

    for i in range(3):
        response = test_model(model, tokenizer, prompt)
        responses.append(response)
        print(f"--- RESPONSE {i+1} ---")
        print(response)
        print("-" * 40)

    # Simple rating interface
    print("\n🔍 EVALUATION TIME")
    print("Rate each response on a 1-5 scale:")
    print("5=Excellent  4=Good  3=Average  2=Poor  1=Very Poor")
    print()

    ratings = []
    for i in range(3):
        while True:
            try:
                rating = int(input(f"Rate Response {i+1} (1-5): "))
                if 1 <= rating <= 5:
                    ratings.append(rating)
                    break
                else:
                    print("Please enter a number from 1 to 5")
            except (ValueError, KeyboardInterrupt):
                print("\nSkipping feedback collection...")
                return responses, [3, 3, 3], [0, 0, 0]  # Default neutral

    # Calculate GRPO advantages
    group_mean = sum(ratings) / len(ratings)
    advantages = [rating - group_mean for rating in ratings]

    print(f"\n📊 GRPO ANALYSIS:")
    print(f"Your ratings: {ratings}")
    print(f"Group baseline: {group_mean:.2f}")
    print(f"Advantages: {[f'{a:+.2f}' for a in advantages]}")
    print("\n💡 Training Impact:")
    for i, adv in enumerate(advantages, 1):
        if adv > 0:
            print(f"  Response {i}: Will be REINFORCED (advantage: {adv:+.2f})")
        elif adv < 0:
            print(f"  Response {i}: Will be DISCOURAGED (advantage: {adv:+.2f})")
        else:
            print(f"  Response {i}: No change (neutral)")

    return responses, ratings, advantages


def apply_grpo_update(model, tokenizer, prompt, responses, advantages):
    if not any(abs(a) > 0.1 for a in advantages):
        print("No significant advantages - skipping policy update")
        return model

    print("\n🔄 Applying GRPO policy updates...")
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE_GRPO)

    for i, (response, advantage) in enumerate(zip(responses, advantages)):
        if abs(advantage) < 0.1:
            continue

        full_text = f"{prompt}\n{response}"
        inputs = tokenizer(
            full_text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH
        )

        if torch.cuda.is_available():
            inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model(**inputs, labels=inputs["input_ids"])
        loss = advantage * outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        print(f"  ✓ Response {i+1}: Updated with advantage {advantage:+.2f}")

    print("Policy update complete!")
    return model


def run_enhanced_grpo_training(model, tokenizer):
    print("=" * 80)
    print("GROUP RELATIVE POLICY OPTIMIZATION (GRPO) - INTERACTIVE LEARNING")
    print("=" * 80)
    print("Mathematical Foundation: A_i = r_i - (1/N) ∑r_j")
    print("Training: Interactive human feedback with group baselines")
    print("=" * 80)

    cleanup_memory()

    scenarios = create_grpo_scenarios()
    print(f"\n🎯 Interactive Training: {len(scenarios)} scenarios")
    print("Process: Generate → Rate → Learn from your preferences")

    completed = 0

    for i, scenario in enumerate(scenarios):
        print(f"\n{'='*20} SESSION {i+1}/{len(scenarios)} {'='*20}")

        try:
            responses, ratings, advantages = interactive_feedback(
                model, tokenizer, scenario
            )
            model = apply_grpo_update(model, tokenizer, scenario, responses, advantages)
            completed += 1

            if i < len(scenarios) - 1:
                print("\n" + "-" * 50)
                continue_choice = (
                    input("Continue to next scenario? (y/n): ").strip().lower()
                )
                if continue_choice in ["n", "no"]:
                    print("Training stopped by user.")
                    break

        except KeyboardInterrupt:
            print("\n\nTraining interrupted by user.")
            break
        except Exception as e:
            print(f"Error in session: {e}")
            continue

    print(f"\n✅ GRPO training complete: {completed}/{len(scenarios)} sessions")
    cleanup_memory()
    return model, tokenizer


# Execute Interactive GRPO
print("\n🚀 Starting Interactive GRPO Training")
print("You'll rate model responses to teach it your preferences.")
print("Press Ctrl+C anytime to skip this step.")

try:
    model, tokenizer = run_enhanced_grpo_training(model, tokenizer)
    evaluate_stage(model, tokenizer, "GRPO")
except KeyboardInterrupt:
    print("\n⏭️  GRPO training skipped - continuing to final evaluation")


🚀 Starting Interactive GRPO Training
You'll rate model responses to teach it your preferences.
Press Ctrl+C anytime to skip this step.
GROUP RELATIVE POLICY OPTIMIZATION (GRPO) - INTERACTIVE LEARNING
Mathematical Foundation: A_i = r_i - (1/N) ∑r_j
Training: Interactive human feedback with group baselines

🎯 Interactive Training: 3 scenarios
Process: Generate → Rate → Learn from your preferences

==================== SESSION 1/3 ====================

SCENARIO: How do I cook vegetables without making them mushy?
Generating 3 responses for your evaluation...

--- RESPONSE 1 ---
Perfect vegetable cooking depends on timing, temperature control, and proper seasoning.

Cook in salted boiling water: Toss with fresh herbs and lemon juice while hot. This ensures even cooking.

Preheat oven to 425°F. Cut vegetables into uniform 1-inch pieces for even cooking. Spread in single layer on baking sheet - overcrowding creates steam instead of cooking.

Cook until fork-tender, about 20-30 minutes. Tast

# Final evaluation

In [8]:
# =============================================================================
# FINAL COMPREHENSIVE EVALUATION AND MODEL SAVING
# =============================================================================


def final_comprehensive_evaluation():
    print("=" * 80)
    print("FINAL COMPREHENSIVE EVALUATION")
    print("=" * 80)
    print("🔄 Training Progression: BASE → SFT → KTO → ORPO → GRPO")
    print("🍳 Domain: Cooking advice and culinary techniques")
    print("📊 Evaluation: Consistent test questions across all stages")
    print("=" * 80)

    # Final comprehensive test
    evaluate_stage(model, tokenizer, "FINAL ENHANCED")

    # Save final model
    print(f"\n💾 Saving final enhanced model...")
    os.makedirs("./models/final_cooking_assistant", exist_ok=True)
    model.save_pretrained("./models/final_cooking_assistant")
    tokenizer.save_pretrained("./models/final_cooking_assistant")

    # Training summary
    print(f"\n{'='*80}")
    print("🎉 TRAINING COMPLETE - COMPREHENSIVE SUMMARY")
    print(f"{'='*80}")
    print("✅ SFT: Domain adaptation with 12 comprehensive cooking examples")
    print(
        f"   └─ {NUM_EPOCHS_SFT} epochs, effective batch size: {BATCH_SIZE_SFT * GRAD_ACCUM_SFT}"
    )
    print("✅ KTO: Behavioral preference learning with 16 preference pairs")
    print(
        f"   └─ {NUM_EPOCHS_KTO} epochs, effective batch size: {BATCH_SIZE_KTO * GRAD_ACCUM_KTO}"
    )
    print("✅ ORPO: Monolithic SFT+preference optimization with 8 pairs")
    print(
        f"   └─ {NUM_EPOCHS_ORPO} epochs, effective batch size: {BATCH_SIZE_ORPO * GRAD_ACCUM_ORPO}"
    )
    print("✅ GRPO: Interactive human feedback with 3 scenarios")
    print(
        f"   └─ {NUM_EPOCHS_GRPO} epochs, effective batch size: {BATCH_SIZE_GRPO * GRAD_ACCUM_GRPO}"
    )

    print(f"\n📈 Enhanced Parameters:")
    total_epochs = NUM_EPOCHS_SFT + NUM_EPOCHS_KTO + NUM_EPOCHS_ORPO + NUM_EPOCHS_GRPO
    print(f"- Total training epochs: {total_epochs}")
    print(f"- Extended context length: {MAX_LENGTH} tokens")
    print(f"- Increased response length: {MAX_NEW_TOKENS} tokens")
    print(f"- Memory optimized for 8GB CUDA")
    print(f"- Progressive evaluation across {len(TEST_QUESTIONS)} test questions")

    print(f"\n💾 Model saved to: ./models/final_cooking_assistant/")
    print(f"{'='*80}")


# Execute final evaluation
final_comprehensive_evaluation()

FINAL COMPREHENSIVE EVALUATION
🔄 Training Progression: BASE → SFT → KTO → ORPO → GRPO
🍳 Domain: Cooking advice and culinary techniques
📊 Evaluation: Consistent test questions across all stages

FINAL ENHANCED MODEL EVALUATION

Q1: How do I cook pasta perfectly?
----------------------------------------
Answer: Proper pasta cooking depends on texture, color, and time.

Use a fork-twist or spoon technique when starting to bring pasta to a boil. This ensures even cooking and prevents clumping.

For short-grain white rice: Bring to boil, then lift immediately using instant-heat water. For brown rice: Use plenty of cold water.

Use a starchy water like salted hot water (not boiling) for pasta. This ensures proper texture without clumping.

Color is crucial - use a fork-twist method when pasta reaches a rolling boil. This ensures uniformity and proper texture.

For al dente pasta: Taste test frequently in the final minutes. Al dente pasta has a firmness where you can bite it tender but not sq

In [9]:
# =============================================================================
# DEMONSTRATION - TESTING THE ENHANCED MODEL
# =============================================================================


def demonstration_test():
    print("\n" + "=" * 60)
    print("🧪 TESTING FINAL ENHANCED MODEL")
    print("=" * 60)
    print("Testing on new cooking scenarios to demonstrate improvements:")

    new_test_prompts = [
        "How do I prevent my cookies from spreading too much?",
        "What's the best way to reheat leftover pizza?",
        "How do I make my soup more flavorful?",
        "What should I do if my sauce is too thin?",
    ]

    for i, prompt in enumerate(new_test_prompts, 1):
        print(f"\n🔍 Test {i}: {prompt}")
        print("-" * 50)
        response = test_model(model, tokenizer, prompt)
        print(f"🤖 Enhanced Model Response:\n{response}")
        if i < len(new_test_prompts):
            print()


print("🚀 Running demonstration tests...")
demonstration_test()

print(f"\n{'='*80}")
print("🎯 COMPLETE LLM FINE-TUNING GUIDE FINISHED")
print("✨ Your model has been progressively enhanced through 5 advanced techniques!")
print(f"{'='*80}")

🚀 Running demonstration tests...

🧪 TESTING FINAL ENHANCED MODEL
Testing on new cooking scenarios to demonstrate improvements:

🔍 Test 1: How do I prevent my cookies from spreading too much?
--------------------------------------------------
🤖 Enhanced Model Response:
Proper cookie spreading requires attention to temperature and texture.

Use a bowl with ice water, not hot water, for all baked goods. This ensures even cooking and prevents spread.

For bread: Use ice-cold water and salt the water until it reaches a gentle boil. Start mixing when the water reaches a rolling pin's thickness - this creates uniformity.

For pasta: Use cold, but not boiling, water. This prevents lumps and ensures even pasta-making.

For baking sheets: Use paper towels instead of water. Cold hands create more sticky fingers than warm ones.

When you poke dough or use toothpick test, form should be clear and dry. This prevents wetting and spreading.

For pastries: Use fork-tack dough or add extra salt while ho